In [ ]:
import pickle 
from qiskit import QuantumCircuit
from qiskit.circuit.library import CXGate,  PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp


from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_precompute import CommutingGateRouterPrecompute

from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph


In [2]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [3]:
filepath = '/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/test_N3_W4.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, [2,1,1])
hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
ess = ExtendedSwapStrategy.from_grid(n, T)
num_physical_qubits = ess._num_vertices
max_layers = 0

Keeping constraints at times: [0 1 2]


In [4]:
hamiltonian = SparsePauliOp(['IIZZ', 'IZZZ'], [1,2])
ess = ExtendedSwapStrategy.from_grid(2, 2)
num_physical_qubits = ess._num_vertices
max_layers = 0

In [5]:
pm = PassManager(
    [
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouterPrecompute(
            ess,
            max_layers=max_layers,
        ),
        SwapToFinalMapping(),
        InverseCancellation(gates_to_cancel=[CXGate()]),
        CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
        InverseCancellation(gates_to_cancel=[CXGate()]),
    ]
)

In [6]:
qc = QuantumCircuit(num_physical_qubits)
qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
tqc = pm.run(qc)

Max layers needed to apply swap decompose: 0
Gates we cannot directly implement: 0
[]
Transpiling un-implemented gates


In [7]:
tqc.draw(fold=-1)

┌───┐┌───────┐                   ┌───┐
q_0 -> 0 ┤ X ├┤ Rz(2) ├──■─────────────■──┤ X ├
         └─┬─┘└───────┘  │             │  └─┬─┘
q_1 -> 1 ──■─────────────┼─────────────┼────■──
                       ┌─┴─┐┌───────┐┌─┴─┐     
q_2 -> 2 ──────────────┤ X ├┤ Rz(4) ├┤ X ├─────
                       └───┘└───────┘└───┘

In [8]:
print_circuit_info(tqc, 'TQC')

TQC has 4 qubits
TQC has 4 non-local gates and 4 non-local depth
TQC contains ['cx', 'rz'] gates.


In [9]:
with open('/lustre/scratch127/qpg/jc59/circuit_depths/results.precompute.120.pkl', 'rb') as f:
    res = pickle.load(f)
for k, v in res.items():
    print(k, v['default'][::-1], v['rz'][0:3]) #, v['rzz'])

test_N4_W6 (4039, 4882) [1712, 4109, 52]
test_N2_W2 (95, 106) [63, 90, 1]
trivial (305, 360) [175, 250, 8]
test_N3_W4 (514, 605) [226, 440, 17]


In [10]:
qc = res['test_N2_W2']['rz'][4]
qc.draw(fold=-1)

global phase: 6.0664
                                                                                                                                         ┌───┐    ┌───┐                   ┌───┐                                                             ┌───┐                                                                             ┌───┐                                                ┌───┐                                              ┌───┐                                                                                                    ┌───┐                                             ┌───┐          ┌───┐                                       ┌────────────────────┐                 ┌──────────────┐                              ┌───┐┌───────────┐┌───┐          
q_0 -> 0 ───────────────────■────────────────────────────────■──────────────────────■────────────────────────────────────────────────────┤ X ├────┤ X ├───────────────────┤ X ├──■──────────────────────────────────────────────────────────┤ X ├──■──────────────────────────────────────────────────────────────────────────┤ X ├────────────────────────────────────────────────┤ X ├───────■──────────────────────────────────────┤ X ├──■────X─────────────────────────────────────────■──────────────────────────────────────────────────┤ X ├────────────────────────■────────────────────┤ X ├───────X──┤ X ├──────────■─────────────────────────■──┤ U(π/2,π/2,-2.0166) ├──────■──────────┤ U(π/2,0,π/2) ├─────────X────────────────────┤ X ├┤ Rz(1.125) ├┤ X ├──────────
         ┌───────────┐      │                                │                      │                                                    └─┬─┘    └─┬─┘              ┌───┐└─┬─┘  │                                                          └─┬─┘  │                                    ┌───┐                                 └─┬─┘                                           ┌───┐└─┬─┘       │                                      └─┬─┘┌─┴─┐  │                             ┌───┐       │                                                  └─┬─┘┌───┐┌───────────┐    ┌─┴─┐     ┌───────────┐└─┬─┘       │  └─┬─┘        ┌─┴─┐        ┌───────────┐┌─┴─┐└────────────────────┘      │          └──────────────┘         │  ┌───┐┌───────────┐└─┬─┘└───────────┘└─┬─┘┌───┐     
q_1 -> 1 ┤ Rz(8.875) ├──────┼──────────────────────■─────────┼──────────────────────┼─────────────■───────────────────────────────■────────■────────┼────────────────┤ X ├──■────┼────────────────────────────────────────────────────────────┼────┼────■───────────────────────────────┤ X ├───────────────────────────────────┼────────────────────────────■────────────────┤ X ├──■─────────┼────────────────────────────────────────┼──┤ X ├──X────────■────────────────────┤ X ├───────┼────────────────────────────────────────────────────┼──┤ X ├┤ Rz(0.625) ├────┤ X ├─────┤ Rz(0.625) ├──■────■────X────┼──────────┤ X ├────────┤ Rz(1.125) ├┤ X ├──────────■─────────────────┼──────────────────────────────■────X──┤ X ├┤ Rz(1.125) ├──■─────────────────■──┤ X ├─────
         └───────────┘    ┌─┴─┐     ┌───────────┐  │       ┌─┴─┐                  ┌─┴─┐           │                               │                 │      ┌───┐     └─┬─┘┌───┐┌─┴─┐                                                   ┌───┐  │  ┌─┴─┐  │                               └─┬─┘                            ┌───┐  │  ┌───┐                     │                └─┬─┘┌───┐     ┌─┴─┐┌───────────┐                  ┌───┐  │  └───┘           │                    └─┬─┘┌───┐┌─┴─┐                                           ┌───┐  │  └─┬─┘└───────────┘    └───┘     └───┬───┬───┘       │         │  ┌───────┴───┴───────┐└───────────┘└───┘          │               ┌─┴─┐     ┌───────────────────┐  │       └─┬─┘└───────────┘                       └─┬─┘     
q_2 -> 2 ─────────────────┤ X ├─────┤ Rz(0.625) ├──┼────■──┤ X ├──────■───────────┤ X ├───────────┼────────■──────────────────────┼────────■────────■──────┤ X ├───────┼──┤ X ├┤ X ├────────────────────────